In [127]:
"""
======================================================================
Bejeweled Blitz 自動玩家 —— 高分衝分優化版（純標準函式庫，僅限 Windows）
======================================================================
相對上一版的主要改動：
  1. 修正 save_debug 因缺少 _write_png 而當掉的問題
  2. 「等盤面靜止再讀」取代固定等待：連續兩次讀盤相同才出手，不會讀到掉落中的畫面
  3. 用「這步有沒有讓盤面改變」判斷失敗，取代「同一步 12 秒內不准重複」
     （舊做法會把大量正常的好步也一起封鎖）
  4. 移除每 10 秒的重新掃瞄：每一步本來就會重新讀盤，那段只是白白停 0.35 秒
  5. 白色寶石改用比例判定；像素顏色加快取，讀盤更快
  6. possible_moves 只檢查被交換的兩格，不再掃整個盤面
  7. timeBeginPeriod(1)：time.sleep 誤差從約 15ms 降到約 1ms

用法（貼進 notebook 的一個 cell 執行後，每段一個 cell）：
    geo = calibrate()      # 左上角按空白鍵 → 右下角按空白鍵
    save_debug(geo)        # 檢查 debug.png 紅圈是否在寶石中心
    show_board(geo)        # 檢查辨識結果
    run(geo)               # 遊戲真正開始的那一刻按 F8
"""

import copy
import ctypes
import colorsys
import math
import os
import struct
import time
import zlib
from ctypes import wintypes

# ----------------------------------------------------------------------
#  可調參數
# ----------------------------------------------------------------------
N = 8
LETTERS = "WROYGBP?"   # 0白 1紅 2橘 3黃 4綠 5藍 6紫 ?未知
BOTTOM_BIAS = 25       # 越大越偏好底部的步（容易連鎖）；設成負數則偏好頂部（盤面較穩）
WHITE_RATIO = 0.6      # 一格內「白色像素」占比超過這個值才判為白寶石
RETRY_COOLDOWN = 1.5   # 某步沒生效後，暫時不再嘗試的秒數（第 n 次失敗就等 1.5*n 秒）
CONFIRM_DELAY = 0.15   # 判定失敗前再多等一下重讀（避免偵測太快、動畫還沒開始就誤判）
STABLE_TIMEOUT = 1.2   # 等盤面靜止的最長秒數（18:15 那版的設定，選步品質較好）
STABLE_TOL = 1         # 兩次讀盤差異 ≤ 這個格數才視為靜止（越小越嚴格，越不會在動畫中途選步）

# ======================================================================
#  1. Windows API 底層設定與 DPI 自動對齊
# ======================================================================
user32 = ctypes.windll.user32
gdi32 = ctypes.windll.gdi32

try:
    ctypes.windll.shcore.SetProcessDpiAwareness(2)
except Exception:
    try:
        user32.SetProcessDPIAware()
    except Exception:
        pass

user32.GetDC.restype = wintypes.HDC
user32.GetDC.argtypes = [wintypes.HWND]
user32.ReleaseDC.argtypes = [wintypes.HWND, wintypes.HDC]
gdi32.CreateCompatibleDC.restype = wintypes.HDC
gdi32.CreateCompatibleDC.argtypes = [wintypes.HDC]
gdi32.CreateCompatibleBitmap.restype = wintypes.HBITMAP
gdi32.CreateCompatibleBitmap.argtypes = [wintypes.HDC, ctypes.c_int, ctypes.c_int]
gdi32.SelectObject.restype = wintypes.HGDIOBJ
gdi32.SelectObject.argtypes = [wintypes.HDC, wintypes.HGDIOBJ]
gdi32.BitBlt.restype = wintypes.BOOL
gdi32.BitBlt.argtypes = [wintypes.HDC, ctypes.c_int, ctypes.c_int, ctypes.c_int,
                         ctypes.c_int, wintypes.HDC, ctypes.c_int, ctypes.c_int,
                         wintypes.DWORD]
gdi32.GetDIBits.argtypes = [wintypes.HDC, wintypes.HBITMAP, wintypes.UINT,
                            wintypes.UINT, ctypes.c_void_p, ctypes.c_void_p,
                            wintypes.UINT]
gdi32.DeleteObject.argtypes = [wintypes.HGDIOBJ]
gdi32.DeleteDC.argtypes = [wintypes.HDC]


class BITMAPINFOHEADER(ctypes.Structure):
    _fields_ = [("biSize", wintypes.DWORD), ("biWidth", wintypes.LONG),
                ("biHeight", wintypes.LONG), ("biPlanes", wintypes.WORD),
                ("biBitCount", wintypes.WORD), ("biCompression", wintypes.DWORD),
                ("biSizeImage", wintypes.DWORD), ("biXPelsPerMeter", wintypes.LONG),
                ("biYPelsPerMeter", wintypes.LONG), ("biClrUsed", wintypes.DWORD),
                ("biClrImportant", wintypes.DWORD)]


class POINT(ctypes.Structure):
    _fields_ = [("x", ctypes.c_long), ("y", ctypes.c_long)]


class Shot:
    """一張截圖。raw 是 BGRA 位元組，像素 (x, y) 的起點是 (y*w + x)*4。"""
    def __init__(self, raw, w, h):
        self.raw, self.w, self.h = raw, w, h


def screen_size():
    return user32.GetSystemMetrics(0), user32.GetSystemMetrics(1)


def grab(left, top, w, h):
    hdc = user32.GetDC(None)
    mdc = gdi32.CreateCompatibleDC(hdc)
    bmp = gdi32.CreateCompatibleBitmap(hdc, w, h)
    old = gdi32.SelectObject(mdc, bmp)
    gdi32.BitBlt(mdc, 0, 0, w, h, hdc, left, top, 0x40CC0020)  # SRCCOPY | CAPTUREBLT
    bi = BITMAPINFOHEADER()
    bi.biSize = ctypes.sizeof(BITMAPINFOHEADER)
    bi.biWidth, bi.biHeight = w, -h
    bi.biPlanes, bi.biBitCount, bi.biCompression = 1, 32, 0
    buf = ctypes.create_string_buffer(w * h * 4)
    gdi32.GetDIBits(mdc, bmp, 0, h, buf, ctypes.byref(bi), 0)
    gdi32.SelectObject(mdc, old)
    gdi32.DeleteObject(bmp)
    gdi32.DeleteDC(mdc)
    user32.ReleaseDC(None, hdc)
    return Shot(buf.raw, w, h)


def cursor():
    p = POINT()
    user32.GetCursorPos(ctypes.byref(p))
    return p.x, p.y


def smooth_drag(x1, y1, x2, y2):
    user32.SetCursorPos(int(x1), int(y1))
    time.sleep(0.008)
    user32.mouse_event(0x0002, 0, 0, 0, 0)  # 左鍵按下
    time.sleep(0.012)

    steps = 3
    for i in range(1, steps + 1):
        cx = x1 + (x2 - x1) * (i / steps)
        cy = y1 + (y2 - y1) * (i / steps)
        user32.SetCursorPos(int(cx), int(cy))
        time.sleep(0.018)

    user32.mouse_event(0x0004, 0, 0, 0, 0)  # 左鍵放開
    time.sleep(0.008)


def _set_timer_resolution(on):
    """Windows 預設計時器精度約 15.6ms，time.sleep(0.008) 實際會睡到 15ms。
    開到 1ms 之後，上面那些短暫停頓才會真的是你寫的長度。"""
    try:
        if on:
            ctypes.windll.winmm.timeBeginPeriod(1)
        else:
            ctypes.windll.winmm.timeEndPeriod(1)
    except Exception:
        pass


def _key_down(vk):
    return bool(user32.GetAsyncKeyState(vk) & 0x8000)


def should_stop():
    """按 Esc，或把滑鼠甩到螢幕任一角落，就停止。"""
    if _key_down(0x1B):
        return True
    x, y = cursor()
    W, H = screen_size()
    return (x <= 2 or x >= W - 3) and (y <= 2 or y >= H - 3)


def _beep():
    try:
        import winsound
        winsound.Beep(880, 120)
    except Exception:
        pass


def _wait_space(prompt):
    print(prompt)
    while _key_down(0x20):
        time.sleep(0.02)
    while not _key_down(0x20):
        if _key_down(0x1B):
            raise KeyboardInterrupt("已取消定位")
        time.sleep(0.01)
    pos = cursor()
    _beep()
    while _key_down(0x20):
        time.sleep(0.02)
    return pos


def calibrate():
    x1, y1 = _wait_space("👉 請將滑鼠移到【左上角 (0,0) 寶石中心】，然後按空白鍵…")
    print(f"✅ 已記錄左上角：({x1}, {y1})")
    x2, y2 = _wait_space("👉 請將滑鼠移到【右下角 (7,7) 寶石中心】，然後按空白鍵…")
    print(f"✅ 已記錄右下角：({x2}, {y2})")

    dx = (x2 - x1) / (N - 1)
    dy = (y2 - y1) / (N - 1)
    if dx < 15 or dy < 15:
        raise ValueError("❌ 定位距離過短，請確認是否有分別對準左上角與右下角！")
    print(f"✅ 標定成功！單格寬高估算：{dx:.1f} x {dy:.1f} px")
    return x1, y1, dx, dy


def center(geo, r, c):
    x0, y0, dx, dy = geo
    return x0 + c * dx, y0 + r * dy


# ---------------- 除錯：不靠任何套件存 PNG ----------------
def _write_png(path, w, h, bgra):
    rows = bytearray()
    for y in range(h):
        row = bgra[y * w * 4:(y + 1) * w * 4]
        rgb = bytearray(w * 3)
        rgb[0::3], rgb[1::3], rgb[2::3] = row[2::4], row[1::4], row[0::4]
        rows += b"\x00" + rgb

    def chunk(tag, data):
        return (struct.pack(">I", len(data)) + tag + data
                + struct.pack(">I", zlib.crc32(tag + data) & 0xFFFFFFFF))

    png = (b"\x89PNG\r\n\x1a\n"
           + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
           + chunk(b"IDAT", zlib.compress(bytes(rows)))
           + chunk(b"IEND", b""))
    with open(path, "wb") as f:
        f.write(png)


def save_debug(geo, path="debug.png"):
    x0, y0, dx, dy = geo
    W, H = screen_size()
    pad = 30
    left, top = max(0, int(x0 - dx / 2) - pad), max(0, int(y0 - dy / 2) - pad)
    w = min(int(dx * N) + 2 * pad, W - left)
    h = min(int(dy * N) + 2 * pad, H - top)
    shot = grab(left, top, w, h)
    buf = bytearray(shot.raw)
    for r in range(N):
        for c in range(N):
            cx, cy = center(geo, r, c)
            cx, cy = int(round(cx - left)), int(round(cy - top))
            for a in range(0, 360, 15):
                px = cx + int(round(8 * math.cos(math.radians(a))))
                py = cy + int(round(8 * math.sin(math.radians(a))))
                for ox in (-1, 0, 1):
                    for oy in (-1, 0, 1):
                        xx, yy = px + ox, py + oy
                        if 0 <= xx < w and 0 <= yy < h:
                            k = (yy * w + xx) * 4
                            buf[k], buf[k + 1], buf[k + 2] = 0, 0, 255
    _write_png(path, w, h, bytes(buf))
    print("📸 已儲存除錯對焦圖表至：", os.path.abspath(path))


# ======================================================================
#  2. 色彩識別引擎
# ======================================================================
_PIXEL_CACHE = {}


def _classify_pixel_raw(r, g, b):
    v = max(r, g, b)
    if v <= 40:
        return -1                       # 太暗：底板、陰影，不投票

    if (v - min(r, g, b)) * 255 < 55 * v:
        # 低飽和度 = 白或灰。亮的才算白寶石；暗灰多半是底板，不投票
        return 0 if v > 110 else -1

    h = colorsys.rgb_to_hsv(r / 255.0, g / 255.0, b / 255.0)[0] * 360.0
    hs = (h + 15) % 360.0               # 平移色相，避免紅色被 0°/360° 切成兩半
    if hs < 30:
        return 1                        # 紅
    elif hs < 55:
        return 2                        # 橘
    elif hs < 90:
        return 3                        # 黃
    elif hs < 170:
        return 4                        # 綠
    elif hs < 260:
        return 5                        # 藍
    return 6                            # 紫


def _classify_pixel(r, g, b):
    """同樣的顏色會一再出現，用快取省掉大量 colorsys 運算。"""
    key = ((r >> 2) << 12) | ((g >> 2) << 6) | (b >> 2)
    v = _PIXEL_CACHE.get(key)
    if v is None:
        v = _PIXEL_CACHE[key] = _classify_pixel_raw(r, g, b)
    return v


def _cell_counts(shot, xa, xb, ya, yb):
    """回傳 (各類別票數 list[7], 有效總票數)。"""
    raw, W = shot.raw, shot.w
    cnt = [0] * 7
    total = 0
    for y in range(ya, yb, 2):
        base = y * W
        for x in range(xa, xb, 2):
            i = (base + x) * 4
            c = _classify_pixel(raw[i + 2], raw[i + 1], raw[i])
            if c >= 0:
                cnt[c] += 1
                total += 1
    return cnt, total


def _classify_cell(shot, xa, xb, ya, yb):
    cnt, total = _cell_counts(shot, xa, xb, ya, yb)
    if total < 5:
        return -1
    if cnt[0] >= WHITE_RATIO * total:   # 白色像素占大多數才是白寶石
        return 0
    # 否則取最多票的彩色（彩色寶石上的白色反光不會干擾）
    return max(range(1, 7), key=lambda k: cnt[k])


def _cell_box(geo, r, c):
    x0, y0, dx, dy = geo
    return (int((c + 0.3) * dx), int((c + 0.7) * dx),
            int((r + 0.3) * dy), int((r + 0.7) * dy))


def read_board(geo):
    x0, y0, dx, dy = geo
    left, top = int(x0 - dx / 2), int(y0 - dy / 2)
    shot = grab(left, top, int(dx * N), int(dy * N))
    board = [[-1] * N for _ in range(N)]
    unknown = 100
    for r in range(N):
        for c in range(N):
            v = _classify_cell(shot, *_cell_box(geo, r, c))
            if v == -1:                 # 認不出來 → 給唯一值，避免被當成同色
                v = unknown
                unknown += 1
            board[r][c] = v
    return board


def _diff(a, b):
    return sum(1 for ra, rb in zip(a, b) for x, y in zip(ra, rb) if x != y)


def read_stable(geo, timeout=None, gap=0.025):
    """連續兩次讀盤幾乎相同（差異 ≤ STABLE_TOL 格，容忍反光/特效閃爍）才算靜止。
    回傳 (盤面, 是否真的靜止)。逾時就用最後一次讀到的，不再傻等。"""
    if timeout is None:
        timeout = STABLE_TIMEOUT
    t_end = time.time() + timeout
    prev = read_board(geo)
    while time.time() < t_end:
        time.sleep(gap)
        cur = read_board(geo)
        if _diff(prev, cur) <= STABLE_TOL:
            return cur, True
        prev = cur
    return prev, False


def show_board(geo):
    b = read_board(geo)
    print("\n--- 當前辨識盤面 ---")
    for row in b:
        print(" ".join(LETTERS[v] if 0 <= v < 7 else "?" for v in row))
    print("--------------------\n")
    return b


def debug_votes(geo, r, c):
    """某一格辨識不對時，用這個看各顏色的票數，據此調整色相門檻或 WHITE_RATIO。"""
    x0, y0, dx, dy = geo
    left, top = int(x0 - dx / 2), int(y0 - dy / 2)
    shot = grab(left, top, int(dx * N), int(dy * N))
    cnt, total = _cell_counts(shot, *_cell_box(geo, r, c))
    print(f"格 ({r},{c})，有效票數 {total}")
    for name, n in zip("白紅橘黃綠藍紫", cnt):
        print(f"  {name}: {n:4d}  ({n / total:.0%})" if total else f"  {name}: 0")


# ---------------- 遊戲計時條 ----------------
# 位置與顏色是從你的截圖量出來的：計時條在盤面正下方，
# 約在最後一列寶石中心再往下 0.85 格；剩餘時間以青色填滿、由右向左縮短。
TIMER_Y_OFFSET = 0.85   # 計時條中心 = 最後一列中心 + 這個值 × 格高


def read_timer(geo):
    """回傳計時條剩餘比例 0~1（1=剛開始，0=時間到）。找不到青色回 0.0。"""
    x0, y0, dx, dy = geo
    left = int(x0 - 0.45 * dx)
    right = int(x0 + 7.45 * dx)
    ycen = int(y0 + (N - 1 + TIMER_Y_OFFSET) * dy)
    shot = grab(left, ycen - 4, right - left, 9)
    raw, W = shot.raw, shot.w
    best = -1
    for y in (1, 4, 7):                    # 取三列，避開計時條中間的白色數字
        for x in range(W - 1, -1, -1):     # 由右往左找最右邊的青色像素 = 填滿的尾端
            i = (y * W + x) * 4
            b, g, r = raw[i], raw[i + 1], raw[i + 2]
            if b > 170 and g > 130 and r < 90 and b - r > 100:
                if x > best:
                    best = x
                break
    return 0.0 if best < 0 else (best + 1) / W


def show_timer(geo):
    """遊戲進行中執行，確認計時條有被偵測到（數字應該隨時間遞減）。"""
    tf = read_timer(geo)
    print(f"計時條剩餘 {tf:.0%}（約 {tf * 60:.0f} 秒）")
    return tf


# ======================================================================
#  3. 高分演算法：特殊寶石加權 + 連鎖模擬
# ======================================================================
def find_matches_detailed(board):
    """回傳 (水平消除格, 垂直消除格, 水平連線長度們, 垂直連線長度們)"""
    R, C = len(board), len(board[0])
    h_matches, v_matches = set(), set()
    h_lens, v_lens = [], []

    for r in range(R):
        c = 0
        while c < C - 2:
            v = board[r][c]
            if v is not None and v < 7 and v == board[r][c + 1] == board[r][c + 2]:
                k = c
                while k < C and board[r][k] == v:
                    h_matches.add((r, k))
                    k += 1
                h_lens.append(k - c)
                c = k
            else:
                c += 1

    for c in range(C):
        r = 0
        while r < R - 2:
            v = board[r][c]
            if v is not None and v < 7 and v == board[r + 1][c] == board[r + 2][c]:
                k = r
                while k < R and board[k][c] == v:
                    v_matches.add((k, c))
                    k += 1
                v_lens.append(k - r)
                r = k
            else:
                r += 1

    return h_matches, v_matches, h_lens, v_lens


def _score_matches(h_m, v_m, h_lens, v_lens):
    """
    - 5 連（超立方體）：+6000
    - L/T 交叉（星光寶石）：+3500 / 個交叉點
    - 4 連（火焰寶石）：+1500
    - 普通 3 連：+200
    """
    score = 0
    for length in h_lens + v_lens:
        if length >= 5:
            score += 6000
        elif length == 4:
            score += 1500
        else:
            score += 200
    score += 3500 * len(h_m & v_m)
    return score


def gravity(board):
    R, C = len(board), len(board[0])
    for c in range(C):
        col = [board[r][c] for r in range(R) if board[r][c] is not None]
        col = [None] * (R - len(col)) + col
        for r in range(R):
            board[r][c] = col[r]


def resolve_simulation(board):
    """連鎖消除直到沒有 match；連鎖越深倍率越高。新補進的寶石未知，所以只模擬既有寶石。"""
    total, combo = 0, 1
    while True:
        h_m, v_m, h_lens, v_lens = find_matches_detailed(board)
        if not (h_m or v_m):
            return total
        total += _score_matches(h_m, v_m, h_lens, v_lens) * combo
        for r, c in h_m | v_m:
            board[r][c] = None
        gravity(board)
        combo += 1


def _line_len(board, r, c, dr, dc):
    """以 (r,c) 為中心，沿 (dr,dc) 方向（雙向）的連續同色長度，含自己。"""
    R, C = len(board), len(board[0])
    v = board[r][c]
    n = 1
    rr, cc = r + dr, c + dc
    while 0 <= rr < R and 0 <= cc < C and board[rr][cc] == v:
        n += 1
        rr += dr
        cc += dc
    rr, cc = r - dr, c - dc
    while 0 <= rr < R and 0 <= cc < C and board[rr][cc] == v:
        n += 1
        rr -= dr
        cc -= dc
    return n


def _makes_match(board, r, c):
    v = board[r][c]
    if v is None or v >= 7:
        return False
    return _line_len(board, r, c, 0, 1) >= 3 or _line_len(board, r, c, 1, 0) >= 3


def possible_moves(board):
    """列出所有能消除的相鄰交換。交換只會改變兩格，所以只檢查這兩格即可。"""
    R, C = len(board), len(board[0])
    moves = []
    for r in range(R):
        for c in range(C):
            v1 = board[r][c]
            if v1 is None or v1 >= 7:
                continue
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= R or c2 >= C:
                    continue
                v2 = board[r2][c2]
                if v2 is None or v2 >= 7 or v1 == v2:
                    continue
                board[r][c], board[r2][c2] = v2, v1
                ok = _makes_match(board, r, c) or _makes_match(board, r2, c2)
                board[r][c], board[r2][c2] = v1, v2
                if ok:
                    moves.append(((r, c), (r2, c2)))
    return moves


def simulate(board, move):
    b = [row[:] for row in board]
    (r1, c1), (r2, c2) = move
    b[r1][c1], b[r2][c2] = b[r2][c2], b[r1][c1]
    return b, resolve_simulation(b)


def choose_move(board, is_blocked=None):
    """預估爆破得分 + 位置加權（BOTTOM_BIAS）。is_blocked(mv) 為 True 的步會被跳過。"""
    best, best_score = None, float("-inf")
    for mv in possible_moves(board):
        if is_blocked is not None and is_blocked(mv):
            continue
        _, gained = simulate(board, mv)
        row = max(mv[0][0], mv[1][0])
        score = gained + row * BOTTOM_BIAS
        if score > best_score:
            best, best_score = mv, score
    return best


def _cells_unchanged(new, old, mv):
    """這步的兩個格子，前後讀到的寶石一樣 = 交換沒生效（無效交換會被遊戲彈回來）。"""
    (r1, c1), (r2, c2) = mv
    return new[r1][c1] == old[r1][c1] and new[r2][c2] == old[r2][c2]


def _is_blocked(tried, board, mv):
    """嘗試過、沒生效、而且那兩格的寶石到現在都沒變 → 冷卻期間內不再嘗試。
    key 含兩格的顏色，所以只要那兩格換成別的寶石，自動解除封鎖。"""
    (r1, c1), (r2, c2) = mv
    rec = tried.get((mv, board[r1][c1], board[r2][c2]))
    if rec is None:
        return False
    count, t = rec
    return time.time() - t < RETRY_COOLDOWN * count


def do_move(geo, mv):
    (r1, c1), (r2, c2) = mv
    x1, y1 = center(geo, r1, c1)
    x2, y2 = center(geo, r2, c2)
    smooth_drag(x1, y1, x2, y2)


# ======================================================================
#  4. 主控循環
# ======================================================================
def run(geo, seconds=75.0, settle=0.22, start_key=0x77, verbose=True, max_fail_streak=30):
    """
    seconds：從按下 F8 起算的最長執行時間，只是保險上限。
             真正的結束時間由遊戲計時條決定（歸零就停手），所以 F8 早按晚按都不會
             造成「程式比遊戲早結束」。
    settle ：每步之後至少等這麼久再讀盤（讓交換動畫先開始）。之後由 read_stable 等到掉落結束。
    start_key：等你按這個鍵（預設 F8）才開始；None 則 3 秒後自動開始。
    verbose：每次有步沒生效、改試其他步時印出提示。
    max_fail_streak：連續這麼多步都沒生效才停手（通常代表已結束或定位跑掉）。
    """
    if start_key is None:
        print("⌛ 3 秒後自動啟動...")
        time.sleep(3)
    else:
        print("🎮 準備就緒！請先在遊戲中按下 Start，倒數『結束、寶石能動』的那一刻，按下【F8】開始！（Esc 取消）")
        while _key_down(start_key):
            time.sleep(0.02)
        while not _key_down(start_key):
            if _key_down(0x1B):
                print("❌ 已手動取消啟動。")
                return
            time.sleep(0.01)

    print("🚀 衝分模式啟動！（Esc 或把滑鼠甩到角落可停止）")
    _set_timer_resolution(True)
    t0 = time.time()
    t_end = t0 + seconds

    moves = reads = idle = fails = unstable = 0
    reason = "⏱️ 時間到"
    tried = {}                 # (步, 格1顏色, 格2顏色) -> (失敗次數, 最後失敗時間)
    last_board = last_mv = None
    fail_streak = bad_reads = idle_streak = 0
    per10 = [0] * 7            # 每 10 秒各執行幾步，用來看程式是從第幾秒開始不動的
    last_move_t = 0.0
    timer_seen = False         # 是否曾偵測到計時條（確認偵測有效後，才用它判斷遊戲結束）
    low_timer = 0
    warned = False

    try:
        while time.time() < t_end:
            if should_stop():
                reason = "🛑 手動按 Esc 或甩滑鼠觸發緊急停止"
                break

            # ---- 遊戲計時條：歸零就停手；偵測不到就提醒一次 ----
            tf = read_timer(geo)
            if tf > 0.3:
                timer_seen = True
            if timer_seen:
                if tf < 0.015:
                    low_timer += 1
                    if low_timer >= 2:
                        reason = "⏳ 遊戲計時條歸零，自動停手"
                        break
                else:
                    low_timer = 0
            elif not warned and time.time() - t0 > 5:
                warned = True
                print("⚠ 5 秒內沒偵測到計時條，將只靠時間上限結束（可用 show_timer(geo) 檢查）")

            board, ok = read_stable(geo)
            reads += 1
            if not ok:
                unstable += 1

            # 讀不到正常盤面（例如被視窗擋住、或已到結算畫面）
            unknowns = sum(1 for row in board for v in row if v >= 7)
            if unknowns > 20:
                bad_reads += 1
                if bad_reads >= 40:
                    reason = "🏁 持續讀不到盤面（可能已結算或視窗被擋住），自動停手"
                    break
                continue
            bad_reads = 0

            # ---- 驗證上一步有沒有生效 ----
            if last_mv is not None:
                if _cells_unchanged(board, last_board, last_mv):
                    # 偵測太快：動畫可能還沒開始，多等一下再讀一次才下結論
                    time.sleep(CONFIRM_DELAY)
                    board, ok = read_stable(geo)
                    reads += 1
                if _cells_unchanged(board, last_board, last_mv):
                    (r1, c1), (r2, c2) = last_mv
                    key = (last_mv, last_board[r1][c1], last_board[r2][c2])
                    count = tried.get(key, (0, 0))[0] + 1
                    tried[key] = (count, time.time())
                    fails += 1
                    fail_streak += 1
                    if verbose:
                        print(f"↩ 第 {count} 次沒生效：{last_mv}，改試其他步")
                    if fail_streak >= max_fail_streak:
                        reason = f"⚠️ 連續 {max_fail_streak} 步都沒生效（可能已結束，或定位/辨色有問題）"
                        break
                else:
                    fail_streak = 0
                last_mv = None
            if len(tried) > 300:
                tried.clear()

            # ---- 選步：跳過「試過沒用」的步 ----
            mv = choose_move(board, lambda m: _is_blocked(tried, board, m))
            if mv is None:
                idle += 1
                idle_streak += 1
                if idle_streak >= 20:       # 所有步都被封鎖太久 → 全部重新給機會
                    tried.clear()
                    idle_streak = 0
                time.sleep(0.05)
                continue
            idle_streak = 0

            do_move(geo, mv)
            moves += 1
            last_board, last_mv = board, mv
            last_move_t = time.time() - t0
            per10[min(int(last_move_t // 10), len(per10) - 1)] += 1
            time.sleep(settle)
    finally:
        user32.mouse_event(0x0004, 0, 0, 0, 0)   # 確保滑鼠左鍵一定放開
        _set_timer_resolution(False)

    elapsed = time.time() - t0
    print(f"\n✅ 結束 [{reason}]")
    print(f"⏱ 實際執行 {elapsed:.1f} 秒，最後一步出手在第 {last_move_t:.1f} 秒"
          f"（計時條偵測：{'正常' if timer_seen else '未偵測到'}）")
    print("📈 每 10 秒的步數：" + " | ".join(f"{i*10}-{i*10+10}s: {n}" for i, n in enumerate(per10)))
    print(f"📊 讀盤 {reads} 次 | 執行 {moves} 步 | 沒生效 {fails} 步 | "
          f"找不到可走的步 {idle} 次 | 盤面不穩逾時 {unstable} 次")

In [128]:
geo = calibrate()

👉 請將滑鼠移到【左上角 (0,0) 寶石中心】，然後按空白鍵…
✅ 已記錄左上角：(1238, 448)
👉 請將滑鼠移到【右下角 (7,7) 寶石中心】，然後按空白鍵…
✅ 已記錄右下角：(1658, 865)
✅ 標定成功！單格寬高估算：60.0 x 59.6 px


In [129]:
save_debug(geo)
show_board(geo)

📸 已儲存除錯對焦圖表至： c:\Users\mayni\OneDrive\文件\GitHub\115-programming\game bot\debug.png

--- 當前辨識盤面 ---
P R W R G P W B
P W B R P Y O Y
G Y B O B W R G
O B R R Y G Y R
W W O G G R W W
Y O Y G P Y B Y
Y Y R Y B P B R
R B Y B P O P W
--------------------



[[6, 1, 0, 1, 4, 6, 0, 5],
 [6, 0, 5, 1, 6, 3, 2, 3],
 [4, 3, 5, 2, 5, 0, 1, 4],
 [2, 5, 1, 1, 3, 4, 3, 1],
 [0, 0, 2, 4, 4, 1, 0, 0],
 [3, 2, 3, 4, 6, 3, 5, 3],
 [3, 3, 1, 3, 5, 6, 5, 1],
 [1, 5, 3, 5, 6, 2, 6, 0]]

In [130]:
run(geo)

🎮 準備就緒！請先在遊戲中按下 Start，倒數『結束、寶石能動』的那一刻，按下【F8】開始！（Esc 取消）
🚀 衝分模式啟動！（Esc 或把滑鼠甩到角落可停止）
↩ 第 1 次沒生效：((0, 3), (1, 3))，改試其他步
↩ 第 1 次沒生效：((4, 0), (5, 0))，改試其他步
↩ 第 1 次沒生效：((3, 4), (4, 4))，改試其他步
↩ 第 1 次沒生效：((3, 0), (3, 1))，改試其他步
↩ 第 1 次沒生效：((4, 2), (5, 2))，改試其他步
↩ 第 1 次沒生效：((5, 2), (6, 2))，改試其他步
↩ 第 1 次沒生效：((6, 5), (6, 6))，改試其他步
↩ 第 1 次沒生效：((6, 2), (7, 2))，改試其他步
↩ 第 1 次沒生效：((6, 0), (7, 0))，改試其他步
↩ 第 1 次沒生效：((2, 3), (3, 3))，改試其他步
↩ 第 1 次沒生效：((4, 0), (4, 1))，改試其他步
↩ 第 1 次沒生效：((5, 3), (6, 3))，改試其他步
↩ 第 1 次沒生效：((5, 2), (6, 2))，改試其他步

✅ 結束 [⏳ 遊戲計時條歸零，自動停手]
⏱ 實際執行 63.8 秒，最後一步出手在第 63.6 秒（計時條偵測：正常）
📈 每 10 秒的步數：0-10s: 18 | 10-20s: 19 | 20-30s: 19 | 30-40s: 18 | 40-50s: 17 | 50-60s: 17 | 60-70s: 7
📊 讀盤 143 次 | 執行 115 步 | 沒生效 13 步 | 找不到可走的步 11 次 | 盤面不穩逾時 0 次
